**Reinforcement Learning from Human Feedback(Proximal Policy Optimization (PPO), Q-Learning, Deep Q-Network (DQN))**

In [ ]:
!pip install stable-baselines3 swig "gymnasium[box2d]" datasets bitsandbytes huggingface_hub
!pip uninstall -y torchao
!pip install --no-cache-dir "unsloth[colab-new]"
!pip install vllm --extra-index-url https://download.pytorch.org/whl/cu121

In [ ]:
import os
import re
import gc
import random
import json
import torch
import trl
import inspect
import torch.nn as nn
import torch.optim as optim
import gymnasium as gym
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from stable_baselines3 import PPO
from datasets import load_dataset, Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    AutoModelForSequenceClassification,
)
from trl import RewardTrainer, RewardConfig, GRPOConfig, GRPOTrainer
from trl.experimental.ppo import PPOTrainer, PPOConfig
from difflib import SequenceMatcher
from unsloth import FastLanguageModel
from vllm import SamplingParams

print("TRL version:", trl.__version__)
print("CUDA available:", torch.cuda.is_available())
print(inspect.signature(GRPOConfig.__init__))

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
  torch.cuda.manual_seed_all(SEED)

dtype = torch.float16 if torch.cuda.is_available() else torch.float32
def clear_memory():
  gc.collect()
  if torch.cuda.is_available():
    torch.cuda.empty_cache()

clear_memory()

In [ ]:
actions = ["left", "right"]
alpha = 0.1
Q = {
    "left": 0,
    "right": 0
}

def get_reward(action):
  if action == "left":
    return 1
  else:
    return -1

for i in range(50):
  action = random.choice(actions)
  reward = get_reward(action)
  Q[action] = Q[action] + alpha * (reward - Q[action])
  print(f"Step {i}, Action: {action}, Reward: {reward}, Q: {Q}")

In [ ]:
Q[action] = Q[action] + alpha * (reward - Q[action])
Q

In [ ]:
class DQN(nn.Module):
  def __init__(self, state_size, action_size):
    super().__init__()
    self.net = nn.Sequential(
        nn.Linear(state_size, 128),
        nn.ReLU(),
        nn.Linear(128, action_size)
    )

  def forward(self, state):
    return self.net(state)

state_size = 4
action_size = 2

model= DQN(state_size, action_size)
target_model = DQN(state_size, action_size)
target_model.load_state_dict(model.state_dict())
optimizer = optim.Adam(model.parameters(), lr=1e-3)
loss_fn = nn.MSELoss()

In [ ]:
state = torch.randn(32, state_size)
next_state = torch.randn(32, state_size)
reward = torch.randn(32)
done = torch.randint(0, 2, (32,)).float()
action = torch.randint(0, action_size, (32,))

q_values = model(state)
q_value = q_values.gather(1, action.unsqueeze(1)).squeeze(1)

with torch.no_grad():
  next_q = target_model(next_state).max(1)[0]
  target = reward + 0.99 * next_q * (1 - done)

loss = loss_fn(q_value, target)
optimizer.zero_grad()
loss.backward()
optimizer.step()

In [ ]:
env = gym.make("CartPole-v1")
state_size = int(env.observation_space.shape[0])
action_size = int(env.action_space.n)

print("State Size:", state_size)
print("Action Size:", action_size)

In [ ]:
gamma = 0.95
epsilon = 1.0
epsilon_decay = 0.99
epsilon_min = 0.01
episodes = 50

model = Sequential([
    Dense(24, activation="relu", input_shape=(state_size,)),
    Dense(action_size, activation="linear")
])

model.compile(loss="mse", optimizer="adam")

In [ ]:
for episode in range(episodes):
  state = env.reset()
  state = state.reshape(1, state_size)

  done = False
  while not done:
    if np.random.rand() < epsilon:
      action = env.action_space.sample()
    else:
      action = np.argmax(model.predict(state, verbose=0))

    next_state, reward, terminated, truncated, _ = env.step(action)
    done = terminated or truncated
    next_state = next_state.reshape(1, state_size)

    target = reward
    if not done:
      target += gamma * np.max(model.predict(next_state, verbose=0))

    q_values = model.predict(state, verbose=0)
    q_values[0][action] = target
    model.fit(state, q_values, epochs=1, verbose=0)
    state = next_state

  epsilon = max(epsilon * epsilon_decay, epsilon_min)
  print(f"Episode: {episode + 1}/{episodes}, Epsilon: {epsilon:.2f}")

In [ ]:
state, _ = env.reset()
state = state.reshape(1, state_size)

total_reward = 0
done = False
while not done:
  action = np.argmax(model.predict(state, verbose=0))
  next_state, reward, terminated, truncated, _ = env.step(action)

  total_reward += reward
  state = next_state.reshape(1, state_size)
  done = terminated or truncated

print("Total Reward:", total_reward)

In [ ]:
env = gym.make("CartPole-v1", render_mode="human")
# Changed from 'CnnPolicy' to 'MlpPolicy' because CartPole observations are vector inputs
model = PPO(
    "MlpPolicy",
    env,
    verbose=1,
    learning_rate=0.0003,
    n_steps=2048,
    batch_size=64,
)

model.learn(total_timesteps=100)
obs, _ = env.reset()
for _ in range(1000):
  action, _ = model.predict(obs)
  # Fixed 'env.stop(action)' to 'env.step(action)'
  obs, reward, done, truncated, info = env.step(action)

  if done or truncated:
    obs, _ = env.reset()

In [ ]:
model_name = "HuggingFaceTB/SmolLM2-135M-Instruct"
tok_rm = AutoTokenizer.from_pretrained(model_name)

if tok_rm.pad_token is None:
  tok_rm.pad_token = tok_rm.eos_token

tok_rm.padding_side = "right"

In [ ]:
loaded_reward_data = []
reward_file_path = "/content/reward_preferences_dataset.jsonl"

try:
    with open(reward_file_path, "r") as f:
      for line in f:
        loaded_reward_data.append(json.loads(line))
    raw_reward_ds = Dataset.from_list(loaded_reward_data)
    print("Total reward rows loaded:", len(raw_reward_ds))
except FileNotFoundError:
    print(f"File {reward_file_path} not found. Ensure it exists or run the dummy block.")

In [ ]:
def make_chat_prompt(user_prompt):
  messages = [
    {"role": "user", "content": user_prompt}
  ]

  return tok_rm.apply_chat_template(
      messages,
      tokenize=False,
      add_generation_prompt=True
  )

def make_reward_example(example):
  return {
      "prompt": make_chat_prompt(example["prompt"]),
      "chosen": example["chosen"] + tok_rm.eos_token,
      "rejected": example["rejected"] + tok_rm.eos_token,
  }

if 'raw_reward_ds' in globals():
    reward_ds = raw_reward_ds.map(
        make_reward_example,
        remove_columns=raw_reward_ds.column_names,
    )
    print("Final RewardTrainer dataset sample:")
    print(reward_ds[0])

In [ ]:
raw_reward_data = [
    {
        "prompt": "How should I prepare for exams?",
        "chosen": "Make a schedule, revise daily, practice previous questions, and sleep well.",
        "rejected": "Do nothing and panic at the end.",
    },
    {
        "prompt": "How do I improve fitness?",
        "chosen": "Exercise consistently, eat balanced meals, hydrate well, and recover properly.",
        "rejected": "Starve yourself, skip sleep, and exercise randomly.",
    },
    {
        "prompt": "What is AI?",
        "chosen": "AI is the ability of machines to perform tasks that usually require human intelligence.",
        "rejected": "AI is random magic with no real meaning.",
    },
    {
        "prompt": "How can I improve my teaching skills?",
        "chosen": "Explain concepts step by step, use examples, ask questions, and summarize key points clearly.",
        "rejected": "Speak fast, ignore students, and make the topic unnecessarily complicated.",
    },
]

reward_data = [make_reward_example(x) for x in raw_reward_data]
reward_ds = Dataset.from_list(reward_data)

print("Reward dataset sample:")
print(reward_ds[0])

In [ ]:
rm = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=1,
    torch_dtype=dtype,
)

rm.config.pad_token_id = tok_rm.pad_token_id
rm.config.use_cache = False

if hasattr(rm, "gradient_checkpointing_enable"):
  rm.gradient_checkpointing_enable()

In [ ]:
rm_args = RewardConfig(
    output_dir="./rm_out",
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    num_train_epochs=1,
    learning_rate=1e-5,
    logging_steps=1,
    eval_strategy="no",
    save_strategy="no",
    report_to="none",
    fp16=False,
    bf16=False,
    max_length=256,
    gradient_checkpointing=True,
    remove_unused_columns=False,
)

rm_trainer = RewardTrainer(
    model=rm,
    args=rm_args,
    train_dataset=reward_ds,
    processing_class=tok_rm,
)

rm_trainer.train()
rm_trainer.save_model("./rm_out")
tok_rm.save_pretrained("./rm_out")

print("Reward Model saved at ./rm_out")

In [ ]:
del rm
del rm_trainer
del tok_rm
clear_memory()

In [ ]:
tok = AutoTokenizer.from_pretrained(model_name)
if tok.pad_token is None:
  tok.pad_token = tok.eos_token

tok.padding_side = "left"

policy = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=dtype,
)

policy.config.pad_token_id = tok.pad_token_id
policy.config.use_cache = False

if hasattr(policy, "gradient_checkpointing_enable"):
  policy.gradient_checkpointing_enable()
if hasattr(policy, "enable_input_require_grads"):
  policy.enable_input_require_grads()

ref_model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=dtype,
)

ref_model.config.pad_token_id = tok.pad_token_id
ref_model.config.use_cache = False
ref_model.eval()

for param in ref_model.parameters():
  param.requires_grad = False

In [ ]:
reward_model = AutoModelForSequenceClassification.from_pretrained(
    "./rm_out",
    num_labels=1,
    torch_dtype=dtype,
)

reward_model.config.pad_token_id = tok.pad_token_id
reward_model.config.use_cache = False
reward_model.eval()

for param in reward_model.parameters():
  param.requires_grad = False

In [ ]:
value_model = AutoModelForSequenceClassification.from_pretrained(
    "./rm_out",
    num_labels=1,
    torch_dtype=dtype,
)

value_model.config.pad_token_id = tok.pad_token_id
value_model.config.use_cache = False

if hasattr(value_model, "gradient_checkpointing_enable"):
  value_model.gradient_checkpointing_enable()

clear_memory()

In [ ]:
prompt_ds = Dataset.from_list([
    {"prompt": "How should I prepare for exams?"},
    {"prompt": "How do I improve fitness?"},
    {"prompt": "What is AI?"},
    {"prompt": "How can I improve my teaching skills?"},
])

def make_ppo_prompt(user_prompt):
  message = [
      {"role": "user", "content": user_prompt}
  ]

  return tok.apply_chat_template(
      message,
      tokenize=False,
      add_generation_prompt=True
  )

def tokenize_for_ppo(example):
  prompt_text = make_ppo_prompt(example["prompt"])
  encoded = tok(
      prompt_text,
      padding=False,
      truncation=True,
      max_length=128,
  )

  return {
      "input_ids": encoded["input_ids"],
      "length": len(encoded["input_ids"]),
  }

ppo_train_dataset = prompt_ds.map(
    tokenize_for_ppo,
    remove_columns=prompt_ds.column_names,
)

ppo_train_dataset = ppo_train_dataset.filter(lambda x: x["length"] <= 128)
print("PPO dataset sample:")
print(ppo_train_dataset[0])

In [ ]:
ppo_args = PPOConfig(
    output_dir="./ppo_out",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=1,
    learning_rate=1e-6,
    num_mini_batches=1,
    num_ppo_epochs=1,
    total_episodes=6,
    response_length=16,
    logging_steps=1,
    save_strategy="no",
    report_to="none",
    fp16=False,
    bf16=False,
    gradient_checkpointing=True,
    remove_unused_columns=False,
    seed=SEED,
)

ppo_trainer = PPOTrainer(
    arg=ppo_args,
    processing_class=tok,
    model=policy,
    ref_model=ref_model,
    reward_model=reward_model,
    value_model=value_model,
    train_dataset=ppo_train_dataset,
)

ppo_trainer.train()
ppo_trainer.save_model("./ppo_out")
tok.save_pretrained("./ppo_out")

In [ ]:
clear_memory()

final_model_path = "./ppo_out"
infer_tok = AutoTokenizer.from_pretrained(final_model_path)
if infer_tok.pad_token is None:
  infer_tok.pad_token = infer_tok.eos_token

infer_tok.padding_side = "left"

infer_model = AutoModelForCausalLM.from_pretrained(
    final_model_path,
    torch_dtype=dtype,
    device_map="auto" if torch.cuda.is_available() else "cpu",
)

infer_model.eval()
test_prompt="How should I prepare for exams?"
messages = [
    {"role": "user", "content": test_prompt}
]

formatted_prompt = infer_tok.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

inputs = infer_tok(
    formatted_prompt,
    return_tensors="pt",
    padding=True,
    truncation=True,
).to(infer_model.device)

with torch.no_grad():
  generated_ids = infer_model.generate(
      **inputs,
      max_new_tokens=80,
      do_sample=True,
      temperature=0.7,
      top_p=0.9,
      pad_token_id=infer_tok.pad_token_id,
      eos_token_id=infer_tok.eos_token_id,
  )

output = infer_tok.batch_decode(generated_ids, skip_special_tokens=True)
print("=== Final Model Output ===")
print(output)

### **GRPO (Group Relative Policy Optimization)**

In [ ]:
data = [
    {
        "prompt": "Explain reinforcement learning in simple terms.",
        "reference_answer": "Reinforcement learning is a type of machine learning where an agent learns by taking actions, receiving rewards or penalties, and improving over time.",
    },
    {
        "prompt": "Write a polite email for delayed delivery.",
        "reference_answer": "We apologize for the delay in your delivery. Your order is on the way, and we appreciate your patience and understanding.",
    },
    {
        "prompt": "What is a neural network?",
        "reference_answer": "A neural network is a machine learning model inspired by the human brain. It learns patterns from data using layers of connected nodes.",
    },
    {
        "prompt": "Explain overfitting in ML.",
        "reference_answer": "Overfitting happens when a model learns the training data too closely, including noise, and performs poorly on new unseen data.",
    },
    {
        "prompt": "Explain supervised learning in simple terms.",
        "reference_answer": "Supervised learning is a machine learning method where a model learns from labeled examples and then predicts outputs for new data.",
    },
    {
        "prompt": "What is gradient descent?",
        "reference_answer": "Gradient descent is an optimization method that helps a model reduce its error by slowly adjusting its parameters in the right direction.",
    },
    {
        "prompt": "Explain classification in machine learning.",
        "reference_answer": "Classification is a machine learning task where the model assigns input data to predefined categories or classes.",
    },
    {
        "prompt": "Write a simple apology message.",
        "reference_answer": "I am sorry for the inconvenience. Thank you for your patience and understanding.",
    },
]

def format_prompt(example):
  return {
      "prompt": f"Question: {example['prompt']}\nAnswer in simple English:"
  }

dataset = Dataset.from_list(data)
dataset = dataset.map(format_prompt)
dataset

In [ ]:
model_id = "HuggingFaceTB/SmolLM-135M-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_id)

if tokenizer.pad_token is None:
  tokenizer.pad_token = tokenizer.eos_token

tokenizer.padding_side = "left"
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto",
)

model.config.pad_token_id = tokenizer.pad_token_id

In [ ]:
lora_config = LoraConfig(
    task_type="CAUSAL_LM",
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
)

def normalize_text(text: str) -> str:
  text = str(text).lower().strip()
  text = re.sub(r"\s+", " ", text)
  return text

def get_completion_text(completion):
  if isinstance(completion, str):
    return completion

  if isinstance(completion, list):
    try:
      return completion[-1]["content"]
    except Exception:
      return str(completion)

  return str(completion)

def similarity_score(a: str, b: str) -> float:
  return SequenceMatcher(None, normalize_text(a), normalize_text(b)).ratio()

def keyword_overlap_score(a: str, b: str) -> float:
  a_words = set(re.findall(r"[a-zA-Z]{3,}", normalize_text(a)))
  b_words = set(re.findall(r"[a-zA-Z]{3,}", normalize_text(b)))

  if not b_words:
    return 0.0

  return len(a_words & b_words) / len(b_words)

def correctness_reward(prompts, completions, reference_answers=None, **kwargs):
  rewards = []
  if reference_answers is None:
    return [0.0 for _ in completions]

  for completions, ref in zip(completions, reference_answers):
    text = get_completion_text(completion)
    sim = similarity_score(text, ref)
    overlap = keyword_overlap_score(text, ref)
    score = (2.5 * sim) + (2.5 * overlap)
    rewards.append(float(score))

  return rewards

def helpfulness_reward(prompts, completions, **kwargs):
  rewards = []
  bad_phrases = [
    "i don't know",
    "no idea",
    "maybe",
    "not sure",
    "random magic",
  ]

  for completion in completions:
    text = get_completion_text(completion)
    text_norm = normalize_text(text)
    score = 0.0

    if 40 <= len(text) <= 260:
      score += 1.0
    elif len(text) < 20:
      score -= 1.0
    elif len(text) > 350:
      score -= 0.5

    if "." in text or "," in text:
      score += 0.5
    if any(bp in text_norm for bp in bad_phrases):
      score -= 1.5

    rewards.append(float(score))

  return rewards

def clarity_reward(prompts, completions, **kwargs):
  rewards = []
  for completion in completions:
    text = get_completion_text(completion)
    text_norm = normalize_text(text)
    score = 0.0

    if re.search(r"[a-zA-Z]", text):
      score += 0.5
    if 30 <= len(text) <= 200:
      score += 0.5
    else:
      score -= 0.5

    if re.search(r"(.)\1{8,}", text_norm):
      score -= 1.0

    rewards.append(float(score))

  return rewards

In [ ]:
training_args = GRPOConfig(
    output_dir="./grpo_out",
    learning_rate=1e-5,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=1,
    num_generations=4,
    max_completion_length=80,
    temperature=0.9, # Fixed missing comma here
    top_p=0.95,
    beta=0.01,
    max_steps=10,
    logging_steps=1,
    save_steps=10,
    remove_unused_columns=False,
    report_to="none",
    gradient_checkpointing=False,
    fp16=torch.cuda_is_available(),
)

trainer = GRPOTrainer(
    model=model,
    processing_class=tokenizer,
    reward_funcs=[
        correctness_reward,
        helpfulness_reward,
        clarity_reward,
    ],
    args=training_args,
    train_dataset=dataset,
    peft_config=lora_config,
)

trainer.train()

In [ ]:
trainer.save_model("grpo_output_final")
tokenizer.save_pretrained("grpo_output_final")

In [ ]:
from peft import PeftModel # Added missing PeftModel import

base_model_id = "HuggingFaceTB/SmolLM-135M-Instruct"
adapter_path = "grpo_output_final"

tok = AutoTokenizer.from_pretrained(adapter_path)
if tok.pad_token is None:
  tok.pad_token = tok.eos_token

base_model = AutoModelForCausalLM.from_pretrained(
    base_model_id,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto",
)

model = PeftModel.from_pretrained(base_model, adapter_path)
model.eval()

In [ ]:
def generate_answer(question, max_new_tokens=100):
  prompt = f"Question: {question}\nAnswer in simple English:",
  inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

  with torch.no_grad():
    outputs = model.generate(
       **inputs,
       max_new_tokens=max_new_tokens,
       do_sample=True,
       temperature=0.7,
       top_p=0.9,
       pad_token_id=tokenizer.eos_token_id,
    )

  return tokenizer.decode(outputs[0], skip_special_tokens=True)

print(generate_answer("Explain AI in simple terms."))

In [ ]:
max_seq_length = 1024
lora_rank = 16
MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"

# Optimized parameters: Lowering gpu_memory_utilization to 0.4 to prevent CUDA OOM crashes in Colab
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=max_seq_length,
    load_in_4bit=True,
    fast_inference=True,
    max_lora_rank=lora_rank,
    gpu_memory_utilization=0.4,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=lora_rank,
    lora_alpha=lora_rank,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    use_gradient_checkpointing="unsloth",
    random_state=3407,
)

In [ ]:
SYSTEM_PROMPT = """
Respond in the following format:


Write your step-by-step reasoning here.


Write only the final numeric answer here.

"""

XML_COT_FORMAT = """\

{reasoning}


{answer}

"""

def extract_hash_answer(text: str):
  if "####" not in text:
    return None
  return text.split("####")[-1].strip()

def get_gsm8k_questions(split="train") -> Dataset:
  data = load_dataset("openai/gsm8k", "main")[split]
  data = data.map(
      lambda x: {
          "prompt": [
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": x["question"]},
            ],
          "answer": extract_hash_answer(x["answer"]),
      }
  )

  return data

dataset = get_gsm8k_questions("train")
print(dataset[0])

In [ ]:
def extract_xml_answer(text: str) -> str:
  # Fixed tags logic to avoid empty string references
  if "<answer>" not in text or "</answer>" not in text:
    return ""

  try:
    answer = text.split("<answer>")[-1]
    answer = answer.split("</answer>")[0]
    return answer.strip()
  except Exception:
    return ""

def correctness_reward_func(prompts, completions, answer, **kwargs) -> list[float]:
  responses = [completion[0]["content"] for completion in completions]
  extracted_responses = [extract_xml_answer(r) for r in responses]

  rewards = []
  for model_answer, gold_answer in zip(extracted_responses, answer):
    if model_answer == gold_answer:
      rewards.append(2.0)
    else:
      rewards.append(0.0)

  return rewards

def int_reward_func(completions, **kwargs) -> list[float]:
  responses = [completion[0]["content"] for completion in completions]
  extracted_responses = [extract_xml_answer(r) for r in responses]

  return [0.5 if r.isdigit() else 0.0 for r in extracted_responses]

def soft_format_reward_func(completions, **kwargs) -> list[float]:
  pattern = r".*?<thought>\s*.*?\s*</thought>\s*<answer>\s*.*?\s*</answer>"
  responses = [completion[0]["content"] for completion in completions]
  matches = [re.match(pattern, r, re.DOTALL) for r in responses]

  return [0.5 if m else 0.0 for m in matches]

def strict_format_reward_func(completions, **kwargs) -> list[float]:
  pattern = r"^\s*<thought>\s*.*?\s*</thought>\s*<answer>\s*.*?\s*</answer>\s*$"
  responses = [completion[0]["content"] for completion in completions]
  matches = [re.match(pattern, r, re.DOTALL) for r in responses]

  return [0.5 if m else 0.0 for m in matches]

def count_xml(text) -> float:
  count = 0.0

  if "<thought>" in text and "</thought>" in text:
    count += 0.125
  if "<answer>" in text and "</answer>" in text:
    count += 0.125
  return count

def xmlcount_reward_func(completions, **kwargs) -> list[float]:
  contents = [completion[0]["content"] for completion in completions]
  return [count_xml(c) for c in contents]

In [ ]:
max_prompt_length = 256

training_args = GRPOConfig(
    output_dir="./grpo_unsloth_out",
    learning_rate=5e-6,
    adam_beta1=0.9,
    adam_beta2=0.99,
    weight_decay=0.1,
    warmup_ratio=0.1,
    lr_scheduler_type="cosine",
    optim="paged_adamw_32bit",
    logging_steps=1,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=1, # Fixed missing value
    num_generations=4,
    max_prompt_length=max_prompt_length,
    max_completion_length=max_seq_length - max_prompt_length,
    max_steps=100,
    save_steps=100,
    max_grad_norm=0.1,
    report_to="none",
)

trainer = GRPOTrainer(
    model=model,
    processing_class=tokenizer,
    reward_funcs=[
        xmlcount_reward_func,
        soft_format_reward_func,
        strict_format_reward_func,
        int_reward_func,
        correctness_reward_func,
    ],
    args=training_args,
    train_dataset=dataset,
)

trainer.train()

In [ ]:
model.save_lora("grpo_unsloth_saved")

In [ ]:
test_prompt = tokenizer.apply_chat_template(
    [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": "If there are 3 boxes and each box has 4 apples, how many apples are there in total?"},
    ],
    tokenize=False,
    add_generation_prompt=True,
)

sampling_params = SamplingParams(
    temperature=0.7,
    top_p=0.95,
    max_tokens=512,
)

output = model.fast_generate(
    test_prompt,
    sampling_params=sampling_params,
    lora_request=model.load_lora("grpo_unsloth_saved"),
)[0].outputs[0].text

print(output)

In [ ]:
model.save_pretrained_merged(
    "grpo_merged_model",
    tokenizer,
    save_method="merged_16bit"
)